# Interview Transcript Extractor (using Groq API which runs on Llama 3.3)

In [ ]:
!pip install groq pandas openpyxl python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv

os.environ["GROQ_API_KEY"] = "hidden for privacy"

if os.environ.get("GROQ_API_KEY"):
    print("SUCCESS")
else:
    print("FAIL")

## State interview questions

In [ ]:
INTERVIEW_QUESTIONS = [
    "What do you do outside of work/school?",
    "How much time would you say you spend on social media on a typical day? What kind of content do you usually see?",
    "How does scrolling through content on social media make you feel?",
    "Do you find yourself comparing your image to people you see online? How does that affect you?",
    "When you're feeling good about yourself, what's usually contributing to that?",
    "What would you say has the biggest influence on how you feel about yourself?",
    "How would you describe your relationship with your body currently?",
    "Have you ever avoided a physical activity because of how you thought you'd look doing working out?",
    "Were you involved in any sports or physical activities? How did that shape how you think about exercise now?",
    "Have you ever signed up for a gym and stopped going? What led to that?",
    "When was the last time you thought about going to the gym but didn't end up going?",
    "If you went to the gym tomorrow, how confident would you be knowing where to start or what to do?",
    "Do you ever feel like gyms are catered for certain types of people? What gives you that impression?",
    "When you think about going to the gym, do you picture yourself going alone or with someone? Why?",
    "What is your preference working out alone, taking a group fitness class, or exercising casually with friends?",
    "If something could remove one barrier to getting you into the gym consistently, what would it be?",
    "How would you feel best supported going to the gym?",
    "What would you hope to gain from going to the gym?",
]

## 4. Define extraction logic and rules

In [ ]:
import json
import time
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

SYSTEM_PROMPT = """You are a qualitative research assistant. You will be given:
1. A list of interview questions.
2. A transcript of an interview in Q&A format (lines starting with "Q:" are questions, lines starting with "A:" are answers).

Your job is to extract the interviewee's answer to each question from the transcript.

IMPORTANT RULES:
- The transcript questions may not match the listed questions exactly. Match by meaning/topic.
- If a question was not addressed at all in the transcript, use "NOT ADDRESSED".
- Paraphrase the interviewee's response concisely (2-4 sentences). Include a key direct quote if one stands out.
- If the interviewee gave a very short or vague answer, note that (e.g., "Brief response: [answer]").
- Be faithful to what the interviewee actually said — do not infer or add meaning.

Respond ONLY with valid JSON. No markdown, no backticks, no preamble. The JSON should be an object where:
- Each key is the full question text (exactly as provided in the list).
- Each value is a string containing the extracted answer or "NOT ADDRESSED".
"""


def build_user_prompt(transcript: str) -> str:
    """Build the user message with questions + transcript."""
    questions_block = "\n".join(
        f"  Q{i+1}. {q}" for i, q in enumerate(INTERVIEW_QUESTIONS)
    )
    return f"""Here are the interview questions:

{questions_block}

Here is the interview transcript:

---
{transcript}
---

Extract the interviewee's answer to each question. Respond with JSON only."""


def extract_answers(transcript: str, max_retries: int = 3) -> dict:
    """Send transcript to Groq/Llama and get structured answers back."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(transcript)},
                ],
                temperature=0.2,
                max_tokens=4096,
                response_format={"type": "json_object"},
            )
            raw = response.choices[0].message.content.strip()
            return json.loads(raw)
        except json.JSONDecodeError as e:
            if attempt < max_retries - 1:
                print(f"JSON parse error, retrying ({attempt + 1}/{max_retries})...")
                time.sleep(2)
            else:
                raise
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 10 * (attempt + 1)
                print(f"Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

## Run

In [ ]:
import glob
import pandas as pd
from pathlib import Path

transcript_dir = Path("transcripts")
if not transcript_dir.exists():
    print("FAILURE")
else:
    transcript_files = sorted(transcript_dir.glob("*.txt"))
    print("SUCCESS")
    records = []
    for i, filepath in enumerate(transcript_files):
        try:
            transcript = filepath.read_text(encoding="utf-8")
            if not transcript.strip():
                print("SKIPPED EMPTY")
                continue
            answers = extract_answers(transcript)
            answers["_source_file"] = filepath.name
            records.append(answers)
            if i < len(transcript_files) - 1:
                time.sleep(3)
        except json.JSONDecodeError as e:
            print("ERROR")
        except Exception as e:
            print("ERROR")
    if records:
        df = pd.DataFrame(records)
        cols = ["_source_file"] + [c for c in df.columns if c != "_source_file"]
        df = df[cols]
    else:
        df = pd.DataFrame()

## Save output to folders and files

In [ ]:
if not df.empty:
    os.makedirs("output", exist_ok=True)
    csv_path = "output/interview_answers.csv"
    df.to_csv(csv_path, index=False)
    json_path = "output/interview_answers.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    xlsx_path = "output/interview_answers.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Answers")
        ws = writer.sheets["Answers"]
        for i, col in enumerate(df.columns):
            max_len = max(df[col].astype(str).map(len).max(), len(col))
            ws.column_dimensions[ws.cell(row=1, column=i+1).column_letter].width = min(max_len + 2, 60)
else:
    print("no data")